# Lab 14 · Agent frameworks: graphs with LangGraph

**Prepared by Mustafa Assaf** · Self-study: Go deeper · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), and save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`). `ask` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit', 'langgraph']):
    %pip install -q --upgrade "agentic-ai-coursekit[framework] @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import make_ask
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
ask = make_ask(provider=PROVIDER)

## Warm-up · Run a tiny graph

A graph connects a node to the next step. This first graph has one deterministic node and uses the real LangGraph library. The later graph adds model generation, checking, and a bounded retry.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class GreetingState(TypedDict):
    message: str

def greet(state):
    return {"message": "Hello, " + state["message"]}

warm_graph = StateGraph(GreetingState)
warm_graph.add_node("greet", greet)
warm_graph.add_edge(START, "greet")
warm_graph.add_edge("greet", END)
print(warm_graph.compile().invoke({"message": "Sam"}))

**Next:** Next, connect generation and checking nodes for an event contract.

## Generate, check, and repair an event contract
**Start with a problem:** A hand-written workflow can become difficult to follow when a failed contract check needs a retry.

**In this lab:** Connect generation and checking as graph nodes, then use a simple Python loop for a bounded repair.

**New:** LangGraph makes workflow state, nodes, and edges explicit. Our graph proposes a change to ShopEasy's `OrderPlaced` event, checks compatibility, and allows one repair.

The rule for this exercise is **additive contract evolution**: keep the existing `orderId` and `total` fields with their types. Add a new optional field rather than renaming `total` to `amount`. This notebook uses a tiny type-map notation, not a complete JSON Schema validator or production schema registry.

**At a glance**

![Lab 14 at a glance: START (OrderPlaced contract), then choose (node), then check (keep old names + types), then Python retry?, then END. Then back to choose: retry · at most 2 attempts.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/14.svg)

## Think about the example

Payment reads the total field in an OrderPlaced event. Renaming or removing that field breaks its existing contract. A new optional field can be added without requiring existing consumers to use it.

**Concepts:** Additive event changes; Published contract; Bounded generator–critic repair.

Use the scenario above to consider these questions before running the code:

1. Why does renaming total break a consumer?
2. Why should a new event field be optional for existing consumers?

## The idea in context

A graph names workflow steps and edges. A simple Python loop retries failed proposals using the check feedback.

An additive event change protects consumers; a framework does not make a proposal compatible automatically.

| Key term | Meaning |
| --- | --- |
| **State** | The data that all steps of a workflow read and write. |
| **Node** | One step of the graph, written as a function of the state. |
| **Edge** | A connection that sets which node runs next. |
| **Retry loop** | Python repeats the graph when its check fails, up to a fixed limit. |

## Pattern · Generator–Critic Loop

A generator proposes; a separate check returns concrete problems; a Python loop retries or stops. A two-attempt cap prevents endless churn. The critic here is deterministic contract checking, not a second model.

## Main exercise at a glance

1. Describe state and two changes
2. Write generation and checking nodes
3. Connect the graph
4. Run, check, and retry

## Step 1 · Describe state and two changes

A `TypedDict` documents fields shared by nodes; it does not validate model output at runtime. `CURRENT` is the published event contract. New fields must use our illustrative `optional string` type so existing consumers are not required to supply them.

`State` names the information passed between nodes. `total=False` allows a run to start with only the project; later nodes add the remaining fields.

In [ ]:
from typing import TypedDict

class State(TypedDict, total=False):
    project: dict
    fields: dict
    attempts: int
    passed: bool
    feedback: str

CURRENT = {"orderId": "string", "total": "number"}
projects = [{"name": "Add coupon", "new_field": "couponCode"},
            {"name": "Add campaign", "new_field": "campaignId"}]

## Step 2 · Write generation and checking nodes
LangGraph requires a function for each node. `choose` sends the current project and feedback to the model; `check` verifies the returned fields. These are the two functions the framework needs, not callbacks you have to build yourself.

The check protects existing field names and types and requires the new field to be optional. It is a small teaching check, not a complete schema validator. A reply that is not JSON becomes a failed check with feedback rather than a crash, so the repair loop can act on prose just like any other bad contract.

In [ ]:
import json

def choose(state):
    project = state["project"]
    attempt = state.get("attempts", 0)
    prompt = (f"Existing event: {CURRENT}. Add {project['new_field']} without breaking consumers. Keep existing names/types. "
              f"For the new field, use the exact type-label text 'optional string', not 'string?'. Reply with only a JSON object, no fences. {state.get('feedback', '')}")
    try:
        fields = json.loads(ask(prompt))
    except json.JSONDecodeError:
        fields = None
    return {"fields": fields, "attempts": attempt + 1}

def check(state):
    fields = state["fields"]
    new = state["project"]["new_field"]
    ok = isinstance(fields, dict)
    if ok:
        for name, expected_type in CURRENT.items():
            if fields.get(name) != expected_type:
                ok = False
        if fields.get(new) != "optional string":
            ok = False
        for name, field_type in fields.items():
            if name not in CURRENT and field_type != "optional string":
                ok = False
    feedback = "" if ok else f"Keep {CURRENT}; the value for {new} must be exactly 'optional string'. Do not rename total."
    return {"passed": ok, "feedback": feedback}

print(check({"project": projects[0], "fields": {"orderId": "string", "amount": "number"}}))

## Step 3 · Connect the graph
Connect the two nodes in order: `START → choose → check → END`. Each graph run makes one proposal and checks it. LangGraph can also route a failed check back to `choose` with a conditional edge inside the graph; this lab keeps the bounded retry in visible Python (Step 4) so the attempt cap stays easy to see.

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(State)
graph.add_node("choose", choose)
graph.add_node("check", check)
graph.add_edge(START, "choose")
graph.add_edge("choose", "check")
graph.add_edge("check", END)
app = graph.compile()
for edge in app.get_graph().edges:
    print(edge.source, "->", edge.target)

## Step 4 · Run, check, and retry
For each project, run the graph once. If the check fails, pass the feedback into one more graph run. The loop stops after a successful check or two attempts, then reports the result.

In [ ]:
results = []
for project in projects:
    state = {"project": project}
    for attempt in range(2):
        state = app.invoke(state)
        if state["passed"]:
            break
    results.append(state)
    print(project["name"], state["fields"], "attempts:", state["attempts"], "passed:", state["passed"])
attempts = 0
for result in results:
    attempts += result["attempts"]
print("Generation attempts:", attempts)
for result in results:
    assert result["passed"], "A contract still breaks after two attempts; stop and review."

## What you should see

The bad contract fails the direct check. For each project, the graph asks the model for a contract and retries at most once if needed. A valid contract retains `orderId: string` and `total: number`, plus the optional new field; attempts vary by response.

## Takeaway

A graph organizes state and steps; the Python loop bounds retries, and the domain checker defines correctness. Additive contracts reduce release coupling, but real compatibility also requires agreed semantics, defaults, versioning, and consumer tests.

## Exercise · Compare a safe addition with a breaking rename

1. In the final `print(check(...))` call in Step 2, set the sample `fields` dictionary to `{"orderId": "string", "amount": "number", "couponCode": "optional string"}`. Rerun Step 2 and read `passed` and `feedback`.
2. In that same sample, rename `amount` back to `total` and keep the new `couponCode` field. Rerun Step 2 again.
3. Expect `passed` to change from `False` to `True`. Explain what the feedback asks the model to preserve when the workflow retries.

**More to try**

- In the same direct check, use `[]` for `fields` and rerun Step 2. Which check rejects it?
- Keep the name `total` and change its type to `"string"`. Rerun Step 2. Why would this break a consumer?
- For your council, put proposal generation and contract review in separate graph nodes. Where should the Python loop escalate a contract that still fails?